In [ ]:
# Simple GenAI app using LangChain
# app is a retrieve information from website and answer questions based on that information
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]= os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# install bs4 -> beautifulsoup4
# install langchain_community
# Data ingestion -> get data from website
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs = loader.load()

In [ ]:
# Divide the data into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
documents = text_splitter.split_documents(docs)

In [ ]:
# Embed the data
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# Vector store - FAISS
# pip install faiss-cpu
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, embeddings)

In [ ]:
# query the data
query = "What is Artificial Intelligence?"
docs = vectorstore.similarity_search(query)
print(docs[0].page_content)

In [ ]:
# llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
# Retrieval chain , document chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Give me answers biased on the following context:
<contexts>
{context}
</contexts>
""") # rather thann searching entire web, we are searching based on the context provided

document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

In [ ]:
from langchain_core.documents import Document

document_chain.invoke({
    "input": "what is Artificial Intelligence?",
    "context": [Document(page_content="Artificial Intelligence (AI) is a field of computer science that aims to create machines capable of intelligent behavior. It encompasses a variety of techniques and approaches, including machine learning, natural language processing, and robotics. AI systems are designed to analyze data, learn from it, and make decisions or predictions based on that information.")]
})# based on the context provided, it will answer the question

In [ ]:
# However, we want the document first come from the retriever we first set up. that way we can use the retriever to dynamically select most relavant documents and pass those for in given question

# so retriever is Input --> Retriever --> Vectore Store
retriever = vectorstore.as_retriever()
from langchain.chains import create_retrieval_chain
retriever_chain = create_retrieval_chain(retriever,document_chain)

In [ ]:
# get the response from the retriever chain
response = retriever_chain.invoke({
    "input": "what is Artificial Intelligence?"
})
print(response)